# 05 — Validação manual do módulo de risco

Este notebook valida quantitativamente as regras de risco **V1** e **V2**.

Até agora:

- o YOLOv8n já foi avaliado no conjunto TEST;
- a regra V1 usa classe + posição horizontal + tamanho;
- a regra V2 acrescenta posição vertical;
- a V2 reduziu alertas excessivos em uma amostra exploratória.

Agora precisamos comparar as duas regras contra um **gabarito manual independente**.

## Objetivo

Construir **50 cenários de validação**, balanceados entre as seis classes tratadas como possíveis obstáculos:

- bed
- sofa
- chair
- table
- wardrobe
- potted plant

Para cada cenário, o avaliador deverá informar:

```text
baixo
medio
alto
```

ou:

```text
excluir
```

quando a detecção estiver errada ou o cenário não puder ser avaliado com segurança.

Depois o notebook calcula automaticamente:

- Accuracy;
- Precision;
- Recall;
- F1-score;
- matriz de confusão;
- desempenho por classe de objeto;
- comparação V1 × V2.

> Esta validação mede o **módulo de risco condicionado a uma detecção do YOLO**.  
> Ela não substitui as métricas de detecção do modelo.

## 1. Organização esperada do projeto

```text
tcc_visao_computacional/
├── models/
│   └── yolov8n_baseline/
│       └── best.pt
│
├── data/
│   └── processed/
│       └── homeobjects_yolo/
│           └── images/
│               └── test/
│
└── reports/
    ├── tables/
    └── validation_risk/
```

## 2. Importações e localização da raiz do projeto

In [ ]:
from pathlib import Path
from collections import defaultdict
import math
import random
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw, ImageFont


def localizar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()

    for candidata in [atual, *atual.parents]:
        if (
            (candidata / "requirements.txt").exists()
            and (candidata / "src").exists()
        ):
            return candidata

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


RAIZ_PROJETO = localizar_raiz_projeto()

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

print("Raiz do projeto:", RAIZ_PROJETO)

## 3. Importar dependências

In [ ]:
try:
    from ultralytics import YOLO
except ImportError:
    import subprocess

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "ultralytics"]
    )

    from ultralytics import YOLO

try:
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
    )
except ImportError:
    import subprocess

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "scikit-learn"]
    )

    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
    )

print("Dependências importadas com sucesso.")

## 4. Caminhos e parâmetros

In [ ]:
CAMINHO_MODELO = (
    RAIZ_PROJETO
    / "models"
    / "yolov8n_baseline"
    / "best.pt"
)

PASTA_TEST = (
    RAIZ_PROJETO
    / "data"
    / "processed"
    / "homeobjects_yolo"
    / "images"
    / "test"
)

PASTA_VALIDACAO = (
    RAIZ_PROJETO
    / "reports"
    / "validation_risk"
)

PASTA_CENARIOS = PASTA_VALIDACAO / "cenarios"
PASTA_TABELAS = RAIZ_PROJETO / "reports" / "tables"

CAMINHO_GABARITO = (
    PASTA_TABELAS
    / "validacao_manual_risco.csv"
)

CAMINHO_RESULTADOS = (
    PASTA_TABELAS
    / "resultados_validacao_risco_v1_v2.csv"
)

QUANTIDADE_CENARIOS = 50
SEMENTE_VALIDACAO = 20260913
CONFIANCA_MINIMA = 0.25
IMGSZ = 640

PASTA_VALIDACAO.mkdir(parents=True, exist_ok=True)
PASTA_CENARIOS.mkdir(parents=True, exist_ok=True)
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

if not CAMINHO_MODELO.exists():
    raise FileNotFoundError(
        f"best.pt não encontrado em:\n{CAMINHO_MODELO}"
    )

if not PASTA_TEST.exists():
    raise FileNotFoundError(
        f"Pasta de TEST não encontrada em:\n{PASTA_TEST}"
    )

print("Modelo:", CAMINHO_MODELO)
print("TEST:", PASTA_TEST)
print("Cenários:", PASTA_CENARIOS)
print("Gabarito:", CAMINHO_GABARITO)

# 5. Regras V1 e V2

Replicamos exatamente as regras utilizadas no notebook anterior.

## V1

```text
peso do objeto
+ posição horizontal
+ tamanho relativo
```

## V2

```text
V1
+ ajuste pela posição vertical
```

In [ ]:
PESO_OBJETO = {
    "bed": 3,
    "sofa": 3,
    "chair": 3,
    "table": 3,
    "wardrobe": 3,
    "potted plant": 2,
}

CLASSES_OBSTACULO = list(PESO_OBJETO.keys())

LIMITE_ESQUERDA = 1 / 3
LIMITE_DIREITA = 2 / 3

LIMITE_AREA_MEDIA = 0.05
LIMITE_AREA_GRANDE = 0.20

LIMITE_VERTICAL_SUPERIOR = 0.60
LIMITE_VERTICAL_INFERIOR = 0.80

PESO_POSICAO = {
    "esquerda": 1,
    "centro": 3,
    "direita": 1,
}

PESO_TAMANHO = {
    "pequeno": 1,
    "medio": 2,
    "grande": 3,
}

AJUSTE_VERTICAL = {
    "superior": -2,
    "intermediaria": -1,
    "inferior": 0,
}

In [ ]:
def classificar_posicao(
    centro_x: float,
    largura_imagem: int,
) -> str:
    centro_normalizado = centro_x / largura_imagem

    if centro_normalizado < LIMITE_ESQUERDA:
        return "esquerda"

    if centro_normalizado < LIMITE_DIREITA:
        return "centro"

    return "direita"


def classificar_tamanho(
    area_box: float,
    area_imagem: float,
):
    proporcao = area_box / area_imagem

    if proporcao < LIMITE_AREA_MEDIA:
        return "pequeno", proporcao

    if proporcao < LIMITE_AREA_GRANDE:
        return "medio", proporcao

    return "grande", proporcao


def classificar_posicao_vertical(
    y2: float,
    altura_imagem: int,
):
    proporcao_vertical = y2 / altura_imagem

    if proporcao_vertical < LIMITE_VERTICAL_SUPERIOR:
        return "superior", proporcao_vertical

    if proporcao_vertical < LIMITE_VERTICAL_INFERIOR:
        return "intermediaria", proporcao_vertical

    return "inferior", proporcao_vertical


def nivel_por_pontuacao(pontuacao: int) -> str:
    if pontuacao <= 4:
        return "baixo"

    if pontuacao <= 6:
        return "medio"

    return "alto"


def classificar_risco_v1(
    classe: str,
    posicao: str,
    tamanho: str,
):
    pontuacao = (
        PESO_OBJETO[classe]
        + PESO_POSICAO[posicao]
        + PESO_TAMANHO[tamanho]
    )

    return pontuacao, nivel_por_pontuacao(pontuacao)


def classificar_risco_v2(
    classe: str,
    posicao: str,
    tamanho: str,
    posicao_vertical: str,
):
    pontuacao_v1, _ = classificar_risco_v1(
        classe=classe,
        posicao=posicao,
        tamanho=tamanho,
    )

    ajuste = AJUSTE_VERTICAL[posicao_vertical]
    pontuacao = max(1, pontuacao_v1 + ajuste)

    return pontuacao, nivel_por_pontuacao(pontuacao), ajuste

## 6. Carregar o modelo

In [ ]:
modelo = YOLO(str(CAMINHO_MODELO))

print("Modelo carregado.")
print(modelo.names)

# 7. Gerar candidatos de validação

Para evitar escolher manualmente apenas casos que pareçam favoráveis,
o notebook executará o modelo sobre **todo o conjunto TEST**.

Em seguida:

- mantém somente as seis classes de possíveis obstáculos;
- seleciona no máximo um cenário por imagem;
- cria uma amostra balanceada entre as seis classes;
- usa uma semente fixa para tornar o processo reproduzível.

Esta etapa pode demorar alguns minutos no computador sem GPU.

In [ ]:
EXTENSOES_IMAGEM = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

imagens_test = sorted(
    arquivo
    for arquivo in PASTA_TEST.rglob("*")
    if arquivo.is_file()
    and arquivo.suffix.lower() in EXTENSOES_IMAGEM
)

print("Imagens no TEST:", len(imagens_test))

In [ ]:
candidatos = []

for indice_imagem, caminho_imagem in enumerate(
    imagens_test,
    start=1,
):
    if (
        indice_imagem == 1
        or indice_imagem % 25 == 0
        or indice_imagem == len(imagens_test)
    ):
        print(
            f"[{indice_imagem:03d}/{len(imagens_test)}] "
            f"{caminho_imagem.name}"
        )

    imagem = Image.open(caminho_imagem).convert("RGB")
    largura, altura = imagem.size
    area_imagem = largura * altura

    resultado = modelo.predict(
        source=str(caminho_imagem),
        conf=CONFIANCA_MINIMA,
        imgsz=IMGSZ,
        verbose=False,
    )[0]

    if resultado.boxes is None:
        continue

    deteccoes_imagem = []

    for box in resultado.boxes:
        classe_id = int(box.cls[0].item())
        classe = modelo.names[classe_id]

        if classe not in CLASSES_OBSTACULO:
            continue

        confianca = float(box.conf[0].item())

        x1, y1, x2, y2 = [
            float(valor)
            for valor in box.xyxy[0].tolist()
        ]

        largura_box = x2 - x1
        altura_box = y2 - y1
        area_box = largura_box * altura_box
        centro_x = (x1 + x2) / 2

        posicao = classificar_posicao(
            centro_x=centro_x,
            largura_imagem=largura,
        )

        tamanho, ocupacao = classificar_tamanho(
            area_box=area_box,
            area_imagem=area_imagem,
        )

        posicao_vertical, proporcao_vertical = (
            classificar_posicao_vertical(
                y2=y2,
                altura_imagem=altura,
            )
        )

        pontuacao_v1, risco_v1 = classificar_risco_v1(
            classe=classe,
            posicao=posicao,
            tamanho=tamanho,
        )

        (
            pontuacao_v2,
            risco_v2,
            ajuste_vertical,
        ) = classificar_risco_v2(
            classe=classe,
            posicao=posicao,
            tamanho=tamanho,
            posicao_vertical=posicao_vertical,
        )

        deteccoes_imagem.append(
            {
                "arquivo": caminho_imagem.name,
                "caminho_imagem": str(caminho_imagem),
                "classe": classe,
                "confianca": confianca,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "posicao": posicao,
                "tamanho": tamanho,
                "ocupacao_imagem": ocupacao,
                "posicao_vertical": posicao_vertical,
                "proporcao_vertical": proporcao_vertical,
                "pontuacao_v1": pontuacao_v1,
                "risco_v1": risco_v1,
                "ajuste_vertical": ajuste_vertical,
                "pontuacao_v2": pontuacao_v2,
                "risco_v2": risco_v2,
            }
        )

    candidatos.extend(deteccoes_imagem)

df_candidatos = pd.DataFrame(candidatos)

print()
print("Detecções candidatas:", len(df_candidatos))

display(
    df_candidatos["classe"]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

# 8. Seleção balanceada de 50 cenários

Queremos aproximadamente a mesma quantidade por classe.

Para aumentar diversidade, o algoritmo prioriza imagens ainda não utilizadas.

In [ ]:
random.seed(SEMENTE_VALIDACAO)

base_por_classe = QUANTIDADE_CENARIOS // len(CLASSES_OBSTACULO)
restante = QUANTIDADE_CENARIOS % len(CLASSES_OBSTACULO)

metas = {
    classe: base_por_classe
    for classe in CLASSES_OBSTACULO
}

for classe in CLASSES_OBSTACULO[:restante]:
    metas[classe] += 1

print("Meta por classe:")
print(metas)

In [ ]:
cenarios_selecionados = []
imagens_utilizadas = set()

for classe in CLASSES_OBSTACULO:
    df_classe = (
        df_candidatos[
            df_candidatos["classe"] == classe
        ]
        .copy()
        .sample(
            frac=1,
            random_state=SEMENTE_VALIDACAO
            + CLASSES_OBSTACULO.index(classe),
        )
    )

    selecionados_classe = []

    # Primeira passagem: uma detecção por imagem.
    for _, linha in df_classe.iterrows():
        if linha["arquivo"] in imagens_utilizadas:
            continue

        selecionados_classe.append(linha)
        imagens_utilizadas.add(linha["arquivo"])

        if len(selecionados_classe) >= metas[classe]:
            break

    # Se faltar, permite reutilizar uma imagem.
    if len(selecionados_classe) < metas[classe]:
        ids_ja_usados = {
            (
                linha["arquivo"],
                linha["x1"],
                linha["y1"],
                linha["x2"],
                linha["y2"],
            )
            for linha in selecionados_classe
        }

        for _, linha in df_classe.iterrows():
            identificador = (
                linha["arquivo"],
                linha["x1"],
                linha["y1"],
                linha["x2"],
                linha["y2"],
            )

            if identificador in ids_ja_usados:
                continue

            selecionados_classe.append(linha)
            ids_ja_usados.add(identificador)

            if len(selecionados_classe) >= metas[classe]:
                break

    cenarios_selecionados.extend(selecionados_classe)

df_cenarios = pd.DataFrame(cenarios_selecionados).reset_index(drop=True)

if len(df_cenarios) < QUANTIDADE_CENARIOS:
    print(
        "AVISO: não foi possível atingir 50 cenários "
        "com os critérios atuais."
    )

df_cenarios.insert(
    0,
    "cenario_id",
    [
        f"C{i:03d}"
        for i in range(1, len(df_cenarios) + 1)
    ],
)

print("Cenários selecionados:", len(df_cenarios))

display(
    df_cenarios["classe"]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

# 9. Gerar imagens para revisão manual

Cada cenário será salvo com:

- ID do cenário;
- classe detectada;
- bounding box alvo destacada;
- informações espaciais;
- **sem mostrar o risco previsto por V1 ou V2**, para não influenciar o avaliador.

Os arquivos ficarão em:

```text
reports/validation_risk/cenarios/
```

In [ ]:
def gerar_imagem_cenario(linha):
    caminho_imagem = Path(linha["caminho_imagem"])
    imagem = Image.open(caminho_imagem).convert("RGB")
    desenho = ImageDraw.Draw(imagem)

    x1 = float(linha["x1"])
    y1 = float(linha["y1"])
    x2 = float(linha["x2"])
    y2 = float(linha["y2"])

    desenho.rectangle(
        [(x1, y1), (x2, y2)],
        width=5,
    )

    texto = (
        f'{linha["cenario_id"]} | '
        f'{linha["classe"]} | '
        f'{linha["posicao"]} | '
        f'{linha["tamanho"]}'
    )

    # Fundo simples para melhorar a legibilidade do texto.
    try:
        bbox_texto = desenho.textbbox((0, 0), texto)
        largura_texto = bbox_texto[2] - bbox_texto[0]
        altura_texto = bbox_texto[3] - bbox_texto[1]
    except Exception:
        largura_texto = len(texto) * 7
        altura_texto = 14

    x_texto = max(0, int(x1))
    y_texto = max(0, int(y1) - altura_texto - 8)

    desenho.rectangle(
        [
            (x_texto, y_texto),
            (
                x_texto + largura_texto + 8,
                y_texto + altura_texto + 8,
            ),
        ],
        fill="black",
    )

    desenho.text(
        (x_texto + 4, y_texto + 4),
        texto,
        fill="white",
    )

    destino = (
        PASTA_CENARIOS
        / f'{linha["cenario_id"]}_{linha["classe"].replace(" ", "_")}.jpg'
    )

    imagem.save(destino, quality=95)

    return destino


caminhos_cenarios = []

for _, linha in df_cenarios.iterrows():
    caminhos_cenarios.append(
        gerar_imagem_cenario(linha)
    )

df_cenarios["imagem_revisao"] = [
    str(caminho)
    for caminho in caminhos_cenarios
]

print(
    f"{len(caminhos_cenarios)} imagens de revisão "
    f"salvas em:\n{PASTA_CENARIOS}"
)

## 10. Visualizar os primeiros cenários

As imagens aparecem sem a previsão V1/V2.

O avaliador deverá julgar somente o potencial do objeto destacado de prejudicar a locomoção.

In [ ]:
def mostrar_cenarios(
    df,
    inicio=0,
    quantidade=10,
    colunas=2,
):
    recorte = df.iloc[
        inicio : inicio + quantidade
    ]

    linhas = math.ceil(len(recorte) / colunas)

    fig, eixos = plt.subplots(
        linhas,
        colunas,
        figsize=(16, linhas * 6),
    )

    if linhas == 1 and colunas == 1:
        eixos = [eixos]
    else:
        eixos = list(eixos.flat)

    for eixo in eixos:
        eixo.axis("off")

    for eixo, (_, linha) in zip(
        eixos,
        recorte.iterrows(),
    ):
        imagem = Image.open(
            linha["imagem_revisao"]
        )

        eixo.imshow(imagem)
        eixo.set_title(
            f'{linha["cenario_id"]} — {linha["classe"]}'
        )
        eixo.axis("off")

    plt.tight_layout()
    plt.show()


mostrar_cenarios(
    df_cenarios,
    inicio=0,
    quantidade=10,
)

# 11. Criar o gabarito manual

Na primeira execução, esta célula cria:

```text
reports/tables/validacao_manual_risco.csv
```

A coluna **`risco_esperado`** ficará vazia.

## Como preencher

Abra o CSV no Excel e, para cada cenário, preencha apenas:

```text
baixo
medio
alto
excluir
```

### Critério sugerido

- **baixo:** pouca ou nenhuma interferência aparente na circulação;
- **medio:** pode exigir desvio ou atenção;
- **alto:** aparente obstrução direta ou forte interferência na trajetória;
- **excluir:** detecção incorreta, objeto ambíguo ou cenário impossível de avaliar.

Não consulte as previsões V1/V2 durante a rotulagem.

> Para uma metodologia ainda mais forte, o ideal é que dois ou mais integrantes façam a classificação de forma independente e depois cheguem a um consenso.

In [ ]:
colunas_gabarito = [
    "cenario_id",
    "arquivo",
    "classe",
    "confianca",
    "posicao",
    "tamanho",
    "ocupacao_imagem",
    "posicao_vertical",
    "proporcao_vertical",
    "imagem_revisao",
    "risco_esperado",
    "observacao",
]

if CAMINHO_GABARITO.exists():
    print(
        "O arquivo de gabarito já existe e NÃO será sobrescrito:"
    )
    print(CAMINHO_GABARITO)
else:
    df_gabarito = df_cenarios.copy()

    df_gabarito["risco_esperado"] = ""
    df_gabarito["observacao"] = ""

    df_gabarito[colunas_gabarito].to_csv(
        CAMINHO_GABARITO,
        index=False,
        encoding="utf-8-sig",
    )

    print("Gabarito criado em:")
    print(CAMINHO_GABARITO)

# ⛔ PARE AQUI NA PRIMEIRA EXECUÇÃO

Agora:

1. abra `reports/tables/validacao_manual_risco.csv`;
2. consulte as imagens em `reports/validation_risk/cenarios/`;
3. preencha `risco_esperado`;
4. salve o CSV;
5. volte ao notebook;
6. execute novamente a partir da próxima seção.

As células abaixo só fazem sentido depois do preenchimento manual.

## 12. Ler e validar o gabarito preenchido

In [ ]:
df_gabarito = pd.read_csv(
    CAMINHO_GABARITO,
    encoding="utf-8-sig",
)

df_gabarito["risco_esperado"] = (
    df_gabarito["risco_esperado"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

ROTULOS_VALIDOS = {
    "baixo",
    "medio",
    "alto",
    "excluir",
}

rotulos_preenchidos = (
    df_gabarito["risco_esperado"] != ""
).sum()

print(
    f"Rótulos preenchidos: "
    f"{rotulos_preenchidos}/{len(df_gabarito)}"
)

valores_invalidos = sorted(
    set(df_gabarito["risco_esperado"])
    - ROTULOS_VALIDOS
    - {""}
)

if valores_invalidos:
    raise ValueError(
        f"Rótulos inválidos encontrados: {valores_invalidos}"
    )

if rotulos_preenchidos < len(df_gabarito):
    faltantes = df_gabarito.loc[
        df_gabarito["risco_esperado"] == "",
        "cenario_id",
    ].tolist()

    raise RuntimeError(
        "Ainda existem cenários sem classificação. "
        f"Preencha: {faltantes}"
    )

print("Gabarito preenchido corretamente.")

## 13. Associar as previsões V1 e V2 ao gabarito

As previsões ficam escondidas durante a rotulagem e são associadas somente agora.

In [ ]:
colunas_previsao = [
    "cenario_id",
    "pontuacao_v1",
    "risco_v1",
    "ajuste_vertical",
    "pontuacao_v2",
    "risco_v2",
]

df_avaliacao = df_gabarito.merge(
    df_cenarios[colunas_previsao],
    on="cenario_id",
    how="left",
    validate="one_to_one",
)

df_avaliacao["usar_na_avaliacao"] = (
    df_avaliacao["risco_esperado"] != "excluir"
)

print(
    "Cenários válidos:",
    int(df_avaliacao["usar_na_avaliacao"].sum()),
)

print(
    "Cenários excluídos:",
    int((~df_avaliacao["usar_na_avaliacao"]).sum()),
)

display(
    df_avaliacao[
        [
            "cenario_id",
            "classe",
            "risco_esperado",
            "risco_v1",
            "risco_v2",
            "usar_na_avaliacao",
        ]
    ].head(10)
)

# 14. Métricas gerais V1 × V2

In [ ]:
df_validos = df_avaliacao[
    df_avaliacao["usar_na_avaliacao"]
].copy()

if len(df_validos) == 0:
    raise RuntimeError(
        "Nenhum cenário válido para avaliação."
    )

y_real = df_validos["risco_esperado"]
y_v1 = df_validos["risco_v1"]
y_v2 = df_validos["risco_v2"]

rotulos_ordem = [
    "baixo",
    "medio",
    "alto",
]

resultados_gerais = []

for nome, previsao in [
    ("V1", y_v1),
    ("V2", y_v2),
]:
    accuracy = accuracy_score(
        y_real,
        previsao,
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            y_real,
            previsao,
            labels=rotulos_ordem,
            average="macro",
            zero_division=0,
        )
    )

    f1_weighted = f1_score(
        y_real,
        previsao,
        labels=rotulos_ordem,
        average="weighted",
        zero_division=0,
    )

    resultados_gerais.append(
        {
            "versao": nome,
            "cenarios_validos": len(df_validos),
            "accuracy": accuracy,
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }
    )

df_metricas_gerais = pd.DataFrame(
    resultados_gerais
)

display(
    df_metricas_gerais.style.format(
        {
            "accuracy": "{:.2%}",
            "precision_macro": "{:.2%}",
            "recall_macro": "{:.2%}",
            "f1_macro": "{:.2%}",
            "f1_weighted": "{:.2%}",
        }
    )
)

## 15. Relatório por nível de risco — V1

In [ ]:
relatorio_v1 = classification_report(
    y_real,
    y_v1,
    labels=["baixo", "medio", "alto"],
    output_dict=True,
    zero_division=0,
)

df_relatorio_v1 = (
    pd.DataFrame(relatorio_v1)
    .T
    .reset_index()
    .rename(columns={"index": "nivel"})
)

display(df_relatorio_v1)

## 16. Relatório por nível de risco — V2

In [ ]:
relatorio_v2 = classification_report(
    y_real,
    y_v2,
    labels=["baixo", "medio", "alto"],
    output_dict=True,
    zero_division=0,
)

df_relatorio_v2 = (
    pd.DataFrame(relatorio_v2)
    .T
    .reset_index()
    .rename(columns={"index": "nivel"})
)

display(df_relatorio_v2)

# 17. Matrizes de confusão

In [ ]:
import numpy as np

matriz_v1 = confusion_matrix(
    y_real,
    y_v1,
    labels=rotulos_ordem,
)

matriz_v2 = confusion_matrix(
    y_real,
    y_v2,
    labels=rotulos_ordem,
)


def mostrar_matriz_confusao(
    matriz,
    titulo,
):
    fig, ax = plt.subplots(figsize=(7, 6))

    imagem = ax.imshow(matriz)

    ax.set_xticks(
        np.arange(len(rotulos_ordem)),
        labels=rotulos_ordem,
    )

    ax.set_yticks(
        np.arange(len(rotulos_ordem)),
        labels=rotulos_ordem,
    )

    ax.set_xlabel("Previsto")
    ax.set_ylabel("Esperado")
    ax.set_title(titulo)

    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            ax.text(
                j,
                i,
                str(matriz[i, j]),
                ha="center",
                va="center",
            )

    fig.colorbar(imagem, ax=ax)
    plt.tight_layout()
    plt.show()


mostrar_matriz_confusao(
    matriz_v1,
    "Matriz de confusão — Risco V1",
)

In [ ]:
mostrar_matriz_confusao(
    matriz_v2,
    "Matriz de confusão — Risco V2",
)

# 18. Acertos por classe de objeto

Além do resultado global, verificamos se alguma classe específica apresenta comportamento ruim.

In [ ]:
df_validos["acerto_v1"] = (
    df_validos["risco_v1"]
    == df_validos["risco_esperado"]
)

df_validos["acerto_v2"] = (
    df_validos["risco_v2"]
    == df_validos["risco_esperado"]
)

desempenho_classe = (
    df_validos
    .groupby("classe")
    .agg(
        cenarios=("cenario_id", "count"),
        accuracy_v1=("acerto_v1", "mean"),
        accuracy_v2=("acerto_v2", "mean"),
    )
    .reset_index()
)

desempenho_classe["diferenca_v2_v1"] = (
    desempenho_classe["accuracy_v2"]
    - desempenho_classe["accuracy_v1"]
)

display(
    desempenho_classe.style.format(
        {
            "accuracy_v1": "{:.2%}",
            "accuracy_v2": "{:.2%}",
            "diferenca_v2_v1": "{:+.2%}",
        }
    )
)

# 19. Casos em que V1 e V2 discordam

Esta seção é especialmente útil para entender se a posição vertical ajudou ou prejudicou.

In [ ]:
df_discordancias = df_validos[
    df_validos["risco_v1"]
    != df_validos["risco_v2"]
].copy()

if df_discordancias.empty:
    print("V1 e V2 não divergiram nos cenários válidos.")
else:
    df_discordancias["v1_correto"] = (
        df_discordancias["risco_v1"]
        == df_discordancias["risco_esperado"]
    )

    df_discordancias["v2_correto"] = (
        df_discordancias["risco_v2"]
        == df_discordancias["risco_esperado"]
    )

    display(
        df_discordancias[
            [
                "cenario_id",
                "classe",
                "posicao",
                "tamanho",
                "posicao_vertical",
                "risco_esperado",
                "risco_v1",
                "risco_v2",
                "v1_correto",
                "v2_correto",
            ]
        ]
    )

    print(
        "Discordâncias:",
        len(df_discordancias),
    )

    print(
        "Casos em que somente V1 acertou:",
        int(
            (
                df_discordancias["v1_correto"]
                & ~df_discordancias["v2_correto"]
            ).sum()
        ),
    )

    print(
        "Casos em que somente V2 acertou:",
        int(
            (
                ~df_discordancias["v1_correto"]
                & df_discordancias["v2_correto"]
            ).sum()
        ),
    )

# 20. Comparação direta e escolha preliminar

In [ ]:
acc_v1 = float(
    df_metricas_gerais.loc[
        df_metricas_gerais["versao"] == "V1",
        "accuracy",
    ].iloc[0]
)

acc_v2 = float(
    df_metricas_gerais.loc[
        df_metricas_gerais["versao"] == "V2",
        "accuracy",
    ].iloc[0]
)

f1_v1 = float(
    df_metricas_gerais.loc[
        df_metricas_gerais["versao"] == "V1",
        "f1_macro",
    ].iloc[0]
)

f1_v2 = float(
    df_metricas_gerais.loc[
        df_metricas_gerais["versao"] == "V2",
        "f1_macro",
    ].iloc[0]
)

print(f"Accuracy V1: {acc_v1:.2%}")
print(f"Accuracy V2: {acc_v2:.2%}")
print()
print(f"F1 macro V1: {f1_v1:.2%}")
print(f"F1 macro V2: {f1_v2:.2%}")
print()

if acc_v2 > acc_v1 and f1_v2 >= f1_v1:
    print(
        "Resultado preliminar: V2 apresentou "
        "desempenho superior nos critérios analisados."
    )
elif acc_v1 > acc_v2 and f1_v1 >= f1_v2:
    print(
        "Resultado preliminar: V1 apresentou "
        "desempenho superior nos critérios analisados."
    )
else:
    print(
        "Resultado misto: Accuracy e F1 não apontam "
        "claramente para a mesma versão. "
        "Analise as matrizes de confusão e os casos de discordância."
    )

## 21. Salvar resultados

In [ ]:
df_metricas_gerais.to_csv(
    PASTA_TABELAS / "metricas_validacao_risco_v1_v2.csv",
    index=False,
    encoding="utf-8-sig",
)

desempenho_classe.to_csv(
    PASTA_TABELAS / "metricas_validacao_risco_por_classe.csv",
    index=False,
    encoding="utf-8-sig",
)

df_avaliacao.to_csv(
    CAMINHO_RESULTADOS,
    index=False,
    encoding="utf-8-sig",
)

if "df_discordancias" in globals():
    df_discordancias.to_csv(
        PASTA_TABELAS / "discordancias_validacao_risco_v1_v2.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Resultados salvos em:")
print(PASTA_TABELAS)

# 22. Interpretação metodológica

Ao final desta etapa teremos duas avaliações diferentes no projeto.

## Detector YOLOv8n

Avaliado no conjunto TEST com métricas de detecção:

- Precision;
- Recall;
- mAP@50;
- mAP@50-95.

## Módulo de risco

Avaliado contra um gabarito humano:

- Accuracy;
- Precision;
- Recall;
- F1-score;
- matriz de confusão.

Isso permite separar claramente:

```text
"O modelo conseguiu detectar o objeto?"
```

de:

```text
"A regra classificou corretamente o impacto na locomoção?"
```

## Limitações

O gabarito manual possui componente subjetivo. Para fortalecer o TCC,
recomenda-se registrar os critérios de rotulagem e, se possível,
utilizar mais de um avaliador com consenso final.

**Próximo passo:** após escolher e congelar a regra final de risco,
implementar a saída por áudio.